In [1]:
import torch
from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
from train import Train
from config import Config
from generate import Generate
from unet import UNet
import matplotlib.pyplot as plt
import time

In [2]:
transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: 2 * x - 1)
    ])

In [3]:
dataset = CIFAR10(root="./cifar10", train=True, transform=transform, download=True)

Files already downloaded and verified


In [4]:
train_data = Subset(dataset, range(100))
train_loader = DataLoader(train_data, batch_size=50, shuffle=True)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
model = UNet(
        in_channels=3,
        down_channels=[16, 32, 64, 128],
        mid_channels=[128, 128, 64],
        up_channels=[128, 64, 32, 16],
        down_sampling=[True, True, False],
        time_embed_dim=64,
        num_down_blocks=2,
        num_mid_blocks=2,
        num_up_blocks=2
    ).to(device)

In [7]:
print(sum((p.numel() for p in model.parameters())))

2731419


In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.MSELoss()

In [9]:
config = Config(
    model_path="./cifar10.pth",
    image_shape=32,
    model=model, 
    train_loader=train_loader, 
    optimizer=optimizer, 
    loss=loss_fn,  
    num_epochs=1,
    in_channels=3,
    learning_rate=0.1e-4,
    num_diffusion_steps=500,
    num_time_steps=500,
    num_steps=500,
    beta_start=1e-4,
    beta_end=0.02,
    device=device
)

In [10]:
trainer = Train(config)

In [11]:
start = time.time()
trainer.fit()
stop = time.time()
print(stop - start)

 ... (more hidden) ...



Epoch: 1 | Loss:  1.4244
Training Process is Finished!
105.28722763061523
